# CNN + LSTM Intrusion Detection System

**TII-SSRC-23 Network Traffic Classification — Multi-Dataset Edition**

This notebook trains a CNN+LSTM model to classify network flows as benign or malicious.
It supports multiple datasets so we can demonstrate results on more than one.

**How to use:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Run all cells in order, OR
3. Edit the `DATASET_NAME` variable in the config cell to switch datasets

**Architecture:** 75 features → Conv1D(16) → BatchNorm → ReLU → LSTM(32) → Dense(64→32→8) → Softmax


## 1. Setup — Install dependencies and check GPU


In [ ]:
# Install required packages
!pip install -q kagglehub imbalanced-learn


In [ ]:
import torch
import numpy as np
import pandas as pd
import os, time, json

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('No GPU — go to Runtime → Change runtime type → GPU')

print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')


## 2. Configuration

**This is the only cell you need to edit when switching datasets.**

Pre-configured datasets:
- `tii-ssrc-23` — Original dataset (8 classes, our primary)
- `cicids-2017` — CIC-IDS-2017 (alternate, 14 attack types)
- `nsl-kdd` — NSL-KDD classic (5 classes)
- `custom` — Upload your own CSV via the Files panel


In [ ]:
# ═══════════════════════════════════════════════════════
# EDIT THIS to switch between datasets
# ═══════════════════════════════════════════════════════
DATASET_NAME = 'tii-ssrc-23'   # Options: 'tii-ssrc-23', 'cicids-2017', 'nsl-kdd', 'custom'

# Training hyperparameters
MODE        = 'multiclass'     # 'multiclass' or 'binary'
EPOCHS      = 20
BATCH_SIZE  = 512
LEARNING_RATE = 0.003
WARMUP_EPOCHS = 2
PATIENCE    = 8
SEED        = 42

# For 'custom' dataset, set the path manually after uploading
CUSTOM_CSV_PATH = '/content/your_dataset.csv'

DATASET_REGISTRY = {
    'tii-ssrc-23': {
        'kaggle_id': 'daniaherzalla/tii-ssrc-23',
        'description': 'TII-SSRC-23 — 8 traffic types (4 benign, 4 malicious)',
    },
    # 'cicids-2017': {
    #     'kaggle_id': 'cicdataset/cicids2017',
    #     'description': 'CIC-IDS-2017 — 14 attack types',
    # },
    'nsl-kdd': {
        'kaggle_id': 'hassan06/nslkdd',
        'description': 'NSL-KDD — Classic IDS benchmark, 5 classes',
    },
}

if DATASET_NAME != 'custom':
    print(f'Selected: {DATASET_NAME}')
    print(f'  → {DATASET_REGISTRY[DATASET_NAME]["description"]}')
else:
    print(f'Custom dataset: {CUSTOM_CSV_PATH}')


## 3. Download dataset

Uses kagglehub to fetch the dataset. First run will download (~5GB for TII-SSRC-23, smaller for others).

**For Kaggle datasets:** You may need to authenticate. Upload your `kaggle.json` credentials file when prompted, or set them as environment variables.


In [ ]:
import kagglehub
import glob

if DATASET_NAME == 'custom':
    csv_path = CUSTOM_CSV_PATH
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f'Custom CSV not found at {csv_path}. Upload it via the Files panel.')
else:
    info = DATASET_REGISTRY[DATASET_NAME]
    print(f'Downloading {info["kaggle_id"]}...')
    download_path = kagglehub.dataset_download(info['kaggle_id'])
    print(f'Downloaded to: {download_path}')

    # Find CSV files in the download
    csv_files = glob.glob(os.path.join(download_path, '**', '*.csv'), recursive=True)
    print(f'\nFound {len(csv_files)} CSV file(s):')
    for f in csv_files:
        size_mb = os.path.getsize(f) / (1024**2)
        print(f'  {size_mb:>8.1f} MB  {os.path.basename(f)}')

    if len(csv_files) == 1:
        csv_path = csv_files[0]
    else:
        # Pick the largest CSV (usually the main one)
        csv_path = max(csv_files, key=os.path.getsize)
        print(f'\nUsing largest: {os.path.basename(csv_path)}')

print(f'\nWill use: {csv_path}')


## 4. Inspect the dataset


In [ ]:
# Peek at first 1000 rows to see structure
preview = pd.read_csv(csv_path, nrows=1000, low_memory=False)
print(f'Columns: {len(preview.columns)}')
print(f'First 5 columns: {list(preview.columns[:5])}')
print(f'Last 5 columns:  {list(preview.columns[-5:])}')
#preview.head(3)


In [ ]:
# NOTES FOR GIVEN BELOW FUNCTION "find_label_columns":

# There are type columns and label columns. Label columns contain the classification of whether attack or benign. Type columns contain the 
# attack type as well. For eg: DDOS type attack or Mirai botnet attacks can be classified by using the type_cols variable in the function defined.

# the line: 
# if type_col is None:
#     type_col = label_col

# means that if there are no attack type columns like DDOS, mirai botnet etc., then we can simply rely on the label column.

# Now this:

# feature_cols = [c for c in preview.columns 
#                 if c not in (label_col, type_col) and 'subtype' not in c.lower()]

# means that Identifyt creates a list of columns that:

# INCLUDED:

# All columns that are NOT:
# label_col
# type_col
# anything containing "subtype"

# EXAMPLE:

# If your dataset has:

# ['duration', 'protocol', 'src_bytes', 'label', 'attack_type']

# Then:

# label_col = 'label'
# type_col = 'attack_type'

# 👉 feature_cols becomes:

# ['duration', 'protocol', 'src_bytes']

In [ ]:
# Identify label columns (works across datasets)
def find_label_columns(columns):
    label_col = None
    type_col = None
    for c in columns:
        cl = c.lower().strip()
        if cl == 'label' or cl == 'class':
            label_col = c
        elif 'type' in cl and 'sub' not in cl:
            type_col = c
        elif cl == 'attack_type' or cl == 'attack':
            type_col = c
    if type_col is None:
        type_col = label_col
    return label_col, type_col

label_col, type_col = find_label_columns(preview.columns)
print(f'Label column: {label_col}')
print(f'Type column:  {type_col}')

feature_cols = [c for c in preview.columns 
                if c not in (label_col, type_col) and 'subtype' not in c.lower()]
print(f'Feature count: {len(feature_cols)}')


## 5. Preprocessing

For large files (>1GB), we stream the CSV in chunks to avoid running out of memory.
For smaller files, we load it all at once.


In [ ]:
# Question: What are standardScaler and LabelEncoder?

# So, standard scaler normalizes numerical data so that all features are on the same scale. The formula for the standard scaler is:
#  z = (x - u) / sigma

#  x = original value
#  u = mean
#  sigma = standard deviation

#  It converts large data into smaller comparable values wrt other column values.

#  For eg:

#  Before scaling:

# Age:        [10, 20, 30]
# Income:     [1000, 50000, 100000]

# 👉 Problem: Income dominates because it's huge.

# After StandardScaler:

# Age:        [-1.22, 0, 1.22]
# Income:     [-1.23, 0, 1.23]

# ✔ Now both features are comparable.



# Now, label encoder converts categorical text into numbers.
# for eg:

# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# labels = ['cat', 'dog', 'dog', 'cat']
# encoded = le.fit_transform(labels)

# result: [0, 1, 1, 0]

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

file_size_gb = os.path.getsize(csv_path) / (1024**3)
print(f'File size: {file_size_gb:.2f} GB')

USE_CHUNKED = file_size_gb > 1.0
print(f'Strategy: {"Chunked streaming" if USE_CHUNKED else "Load all at once"}')


In [ ]:
def preprocess_in_memory(csv_path, label_col, type_col, feature_cols, mode, seed=42):
    """Load entire CSV into memory and preprocess."""
    print('Loading CSV...')
    t0 = time.time()
    df = pd.read_csv(csv_path, low_memory=False)
    print(f'  → Loaded {len(df):,} rows in {time.time()-t0:.1f}s')

    X = df[feature_cols].values.astype(np.float32) # why np.float32 ?
    X = np.nan_to_num(X, nan=0.0, posinf=1e10, neginf=-1e10)

    y_raw = df[type_col if mode == 'multiclass' else label_col].astype(str).values

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    class_names = le.classes_.tolist()

    # Split: 72% train, 8% val, 20% test; why this particular ratio?
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=seed)

    scaler = StandardScaler()
    X_tr  = scaler.fit_transform(X_tr).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_te  = scaler.transform(X_te).astype(np.float32)

    n_feat = X_tr.shape[1] # number of features
    X_tr  = X_tr.reshape(-1, n_feat, 1)
    X_val = X_val.reshape(-1, n_feat, 1) # ?
    X_te  = X_te.reshape(-1, n_feat, 1)

    return {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_te, 'y_test': y_te,
        'scaler': scaler, 'label_encoder': le,
        'class_names': class_names, 'n_classes': len(class_names),
        'n_features': n_feat,
    }

def preprocess_chunked(csv_path, label_col, type_col, feature_cols, mode,
                        chunksize=100_000, sample_for_scaler=200_000, seed=42):
    """Stream CSV in chunks for very large files."""
    print('Phase 1: Scan for row count and label values...')
    n_rows = 0
    label_values = set()
    target_col = type_col if mode == 'multiclass' else label_col
    for chunk in pd.read_csv(csv_path, usecols=[target_col], chunksize=chunksize, low_memory=False):
        n_rows += len(chunk)
        label_values.update(chunk[target_col].astype(str).unique())
    print(f'  → {n_rows:,} rows, {len(label_values)} unique labels')

    le = LabelEncoder()
    le.fit(sorted(label_values))
    class_names = le.classes_.tolist() # what is le.classes_ ?

    print('Phase 2: Sample for scaler fitting...')
    rng = np.random.default_rng(seed) # what is rng?
    reservoir = []
    seen = 0
    for chunk in pd.read_csv(csv_path, usecols=feature_cols, chunksize=chunksize, low_memory=False):
        vals = np.nan_to_num(chunk.values.astype(np.float32), nan=0, posinf=1e10, neginf=-1e10)
        for row in vals:
            seen += 1
            if len(reservoir) < sample_for_scaler:
                reservoir.append(row)
            else:
                j = rng.integers(0, seen)
                if j < sample_for_scaler:
                    reservoir[j] = row
    scaler = StandardScaler().fit(np.array(reservoir, dtype=np.float32))
    print(f'  → Fitted on {len(reservoir):,} samples')

    print('Phase 3: Stream and split into arrays...')
    X_tr_l, y_tr_l = [], []
    X_val_l, y_val_l = [], []
    X_te_l, y_te_l = [], []

    for chunk in pd.read_csv(csv_path, chunksize=chunksize, low_memory=False):
        X = np.nan_to_num(chunk[feature_cols].values.astype(np.float32),
                          nan=0, posinf=1e10, neginf=-1e10)
        X = scaler.transform(X).astype(np.float32)
        y = le.transform(chunk[target_col].astype(str).values)

        # Random split: 72/8/20
        splits = rng.choice(['train', 'val', 'test'], size=len(X), p=[0.72, 0.08, 0.20])
        for s, Xl, yl in [('train', X_tr_l, y_tr_l), ('val', X_val_l, y_val_l), ('test', X_te_l, y_te_l)]:
            mask = splits == s
            if mask.any():
                Xl.append(X[mask])
                yl.append(y[mask])

    n_feat = len(feature_cols)
    X_tr  = np.vstack(X_tr_l).reshape(-1, n_feat, 1)
    X_val = np.vstack(X_val_l).reshape(-1, n_feat, 1)
    X_te  = np.vstack(X_te_l).reshape(-1, n_feat, 1)
    y_tr  = np.concatenate(y_tr_l)
    y_val = np.concatenate(y_val_l)
    y_te  = np.concatenate(y_te_l)

    return {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_te, 'y_test': y_te,
        'scaler': scaler, 'label_encoder': le,
        'class_names': class_names, 'n_classes': len(class_names),
        'n_features': n_feat,
    }


In [ ]:
# Run preprocessing
if USE_CHUNKED:
    data = preprocess_chunked(csv_path, label_col, type_col, feature_cols, MODE, seed=SEED)
else:
    data = preprocess_in_memory(csv_path, label_col, type_col, feature_cols, MODE, seed=SEED)

print(f'\nMode: {MODE}')
print(f'Classes ({data["n_classes"]}): {data["class_names"]}')
print(f'Train: {len(data["X_train"]):,} | Val: {len(data["X_val"]):,} | Test: {len(data["X_test"]):,}')
print(f'Input shape: ({data["n_features"]}, 1)')

# Class distribution
print('\nTraining set class distribution:')
for i, name in enumerate(data['class_names']):
    cnt = (data['y_train'] == i).sum()
    pct = 100 * cnt / len(data['y_train'])
    print(f'  {name:>22s}: {cnt:>8,d} ({pct:5.1f}%)')


## 6. Model — CNN + LSTM Architecture

Same architecture from the local project:
- **Conv1D** (16 filters, kernel 3) — learns local patterns across adjacent features
- **BatchNorm + ReLU** — normalization and nonlinearity
- **LSTM** (32 hidden units) — captures sequential dependencies across the 73 conv outputs
- **Dense layers** (64 → 32 → n_classes) with dropout — final classification


In [ ]:
import torch.nn as nn

class CNN_LSTM_IDS(nn.Module):
    def __init__(self, n_features, n_classes,
                 conv_filters=16, conv_kernel=3,
                 lstm_hidden=32, d1=64, d2=32, dropout=0.2):
        super().__init__()
        self.conv = nn.Conv1d(1, conv_filters, conv_kernel, padding=0)
        nn.init.kaiming_normal_(self.conv.weight, nonlinearity='relu')

        self.bn   = nn.BatchNorm1d(conv_filters)
        self.relu = nn.ReLU()

        self.lstm = nn.LSTM(conv_filters, lstm_hidden, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, d1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d1, d2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d2, n_classes),
        )
        for m in self.classifier:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (batch, n_features, 1)
        h = x.permute(0, 2, 1)         # → (batch, 1, n_features)
        h = self.relu(self.bn(self.conv(h)))
        h = h.permute(0, 2, 1)         # → (batch, conv_out, conv_filters)
        _, (h_n, _) = self.lstm(h)
        h = h_n.squeeze(0)              # final hidden state
        return self.classifier(h)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = CNN_LSTM_IDS(data['n_features'], data['n_classes']).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')


## 7. Training

AdamW optimizer with linear warmup + cosine annealing learning rate schedule.
Early stopping monitors validation accuracy.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# Class weights (sqrt-dampened) for handling imbalance
classes, counts = np.unique(data['y_train'], return_counts=True)
n_classes = data['n_classes']
total = len(data['y_train'])
weight_tensor = torch.ones(n_classes, device=device)
for c, cnt in zip(classes, counts):
    weight_tensor[int(c)] = float(np.sqrt(total / (n_classes * cnt)))

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Build dataloaders
train_ds = TensorDataset(torch.from_numpy(data['X_train']), torch.from_numpy(data['y_train']))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=torch.cuda.is_available())

X_val_t = torch.from_numpy(data['X_val']).to(device)
y_val_t = torch.from_numpy(data['y_val']).to(device)

n_batches = len(train_loader)
total_steps = EPOCHS * n_batches
warmup_steps = WARMUP_EPOCHS * n_batches

def get_lr(step):
    if step < warmup_steps:
        return LEARNING_RATE * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return LEARNING_RATE * 0.5 * (1 + np.cos(np.pi * progress))

# Mixed precision for speed on Colab T4
use_amp = torch.cuda.is_available()
scaler_amp = torch.amp.GradScaler('cuda', enabled=use_amp)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_state = None
patience_counter = 0
global_step = 0

print(f'Training: {EPOCHS} epochs | batch={BATCH_SIZE} | lr={LEARNING_RATE}')
print('=' * 70)

t_start = time.time()

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0

    for xb, yb in train_loader:
        for g in optimizer.param_groups:
            g['lr'] = get_lr(global_step)
        global_step += 1

        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()

        running_loss += loss.item() * xb.size(0)
        running_correct += (logits.argmax(1) == yb).sum().item()
        running_total += xb.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total

    # Validation
    model.eval()
    with torch.no_grad():
        val_logits_parts = []
        for i in range(0, len(X_val_t), 1024):
            with torch.amp.autocast('cuda', enabled=use_amp):
                val_logits_parts.append(model(X_val_t[i:i+1024]))
        val_logits = torch.cat(val_logits_parts, dim=0)
        val_loss = criterion(val_logits.float(), y_val_t.long()).item()
        val_acc = (val_logits.argmax(1) == y_val_t).float().mean().item()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        marker = ' *'
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    print(f'Epoch {epoch+1:>3d}/{EPOCHS}  '
          f'loss:{train_loss:.4f} acc:{train_acc:.4f} | '
          f'v_loss:{val_loss:.4f} v_acc:{val_acc:.4f}  '
          f'({elapsed:.1f}s){marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f'\nRestored best weights (val_acc={best_val_acc:.4f})')

print(f'\nTotal training time: {time.time() - t_start:.1f}s')


## 8. Evaluation


In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, roc_curve, auc)

model.eval()
X_test_t = torch.from_numpy(data['X_test']).to(device)
with torch.no_grad():
    test_logits_parts = []
    for i in range(0, len(X_test_t), 1024):
        test_logits_parts.append(model(X_test_t[i:i+1024]))
    test_logits = torch.cat(test_logits_parts, dim=0)
    test_proba = torch.softmax(test_logits.float(), dim=1).cpu().numpy()
    test_pred = test_logits.argmax(1).cpu().numpy()

y_test = data['y_test']
acc = accuracy_score(y_test, test_pred)
prec, rec, f1, support = precision_recall_fscore_support(y_test, test_pred, average=None, zero_division=0)
macro_f1 = np.mean(f1)
weighted_f1 = np.average(f1, weights=support)

print(f'Dataset: {DATASET_NAME}')
print(f'Mode:    {MODE}')
print(f'='*60)
print(f'Overall Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print(f'Macro F1:         {macro_f1:.4f}')
print(f'Weighted F1:      {weighted_f1:.4f}')
print(f'='*60)
print(f'\n{"Class":>22s}  {"Prec":>8s}  {"Recall":>8s}  {"F1":>8s}  {"Support":>8s}')
print('-' * 60)
for i, name in enumerate(data['class_names']):
    print(f'{name:>22s}  {prec[i]:>8.4f}  {rec[i]:>8.4f}  {f1[i]:>8.4f}  {support[i]:>8d}')


## 9. Visualizations


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('dark_background')

# Training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_x, history['train_loss'], label='Train', color='#58a6ff', lw=2)
ax1.plot(epochs_x, history['val_loss'], label='Val', color='#f78166', lw=2)
ax1.set_title('Loss Over Epochs'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.2)
ax2.plot(epochs_x, history['train_acc'], label='Train', color='#3fb950', lw=2)
ax2.plot(epochs_x, history['val_acc'], label='Val', color='#d2a8ff', lw=2)
ax2.set_title('Accuracy Over Epochs'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.2)
plt.suptitle(f'Training History — {DATASET_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'training_history_{DATASET_NAME}.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, test_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, cmap='RdYlGn', vmin=0, vmax=1)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm_norm[i,j] < 0.5 else 'black'
        ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.1%})',
                ha='center', va='center', color=color, fontsize=9)
ax.set_xticks(range(len(data['class_names'])))
ax.set_yticks(range(len(data['class_names'])))
ax.set_xticklabels(data['class_names'], rotation=45, ha='right')
ax.set_yticklabels(data['class_names'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {DATASET_NAME}', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(f'confusion_matrix_{DATASET_NAME}.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ROC curves (one-vs-rest)
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
for i, name in enumerate(data['class_names']):
    y_bin = (y_test == i).astype(int)
    if test_proba.shape[1] > i and y_bin.sum() > 0:
        fpr, tpr, _ = roc_curve(y_bin, test_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{name} (AUC={roc_auc:.3f})')
ax.plot([0,1], [0,1], 'w--', alpha=0.3)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — {DATASET_NAME}', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'roc_curves_{DATASET_NAME}.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Per-class F1 bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(data['class_names']))
width = 0.25
ax.bar(x - width, prec, width, label='Precision', color='#58a6ff')
ax.bar(x,         rec,  width, label='Recall',    color='#3fb950')
ax.bar(x + width, f1,   width, label='F1',        color='#d2a8ff')
ax.axhline(y=0.9, color='#f0883e', linestyle='--', alpha=0.5, label='90% threshold')
ax.set_xticks(x); ax.set_xticklabels(data['class_names'], rotation=45, ha='right')
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title(f'Per-Class Metrics — {DATASET_NAME}', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig(f'class_metrics_{DATASET_NAME}.png', dpi=120, bbox_inches='tight')
plt.show()


## 10. Save results

Saves the model, metrics, and a results summary to Colab's local storage.
Download these to compare across datasets.


In [ ]:
# Save everything
out_dir = f'/content/results_{DATASET_NAME}'
os.makedirs(out_dir, exist_ok=True)

# Model weights
torch.save(model.state_dict(), f'{out_dir}/model.pth')

# Metrics summary
results = {
    'dataset': DATASET_NAME,
    'mode': MODE,
    'accuracy': float(acc),
    'macro_f1': float(macro_f1),
    'weighted_f1': float(weighted_f1),
    'n_classes': n_classes,
    'class_names': data['class_names'],
    'per_class': {
        name: {
            'precision': float(prec[i]),
            'recall': float(rec[i]),
            'f1': float(f1[i]),
            'support': int(support[i]),
        }
        for i, name in enumerate(data['class_names'])
    },
    'history': {k: [float(x) for x in v] for k, v in history.items()},
    'training_samples': int(len(data['X_train'])),
    'test_samples': int(len(data['X_test'])),
}

with open(f'{out_dir}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save scaler too
import pickle
with open(f'{out_dir}/scaler.pkl', 'wb') as f:
    pickle.dump(data['scaler'], f)

print(f'Saved to {out_dir}/')
print('Files:')
for f in os.listdir(out_dir):
    size = os.path.getsize(os.path.join(out_dir, f)) / 1024
    print(f'  {size:>8.1f} KB  {f}')


## 11. Compare results across datasets

Run cells 2–10 again with a different `DATASET_NAME`, then run this cell to see them side by side.


In [ ]:
# Load all results that have been saved so far
import glob
result_dirs = sorted(glob.glob('/content/results_*'))

if len(result_dirs) < 2:
    print(f'Only {len(result_dirs)} dataset(s) trained so far.')
    print('Edit the DATASET_NAME variable in cell 2 and re-run cells 3-10 with another dataset.')
else:
    print(f'{"Dataset":<20s}  {"Acc":>8s}  {"MacroF1":>8s}  {"Classes":>8s}  {"TestN":>8s}')
    print('-' * 60)
    all_results = []
    for d in result_dirs:
        with open(f'{d}/results.json') as f:
            r = json.load(f)
        all_results.append(r)
        print(f'{r["dataset"]:<20s}  '
              f'{r["accuracy"]:>8.4f}  '
              f'{r["macro_f1"]:>8.4f}  '
              f'{r["n_classes"]:>8d}  '
              f'{r["test_samples"]:>8,d}')

    # Bar chart comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    names = [r['dataset'] for r in all_results]
    accs = [r['accuracy'] for r in all_results]
    f1s  = [r['macro_f1'] for r in all_results]
    x = np.arange(len(names))
    width = 0.35
    ax.bar(x - width/2, accs, width, label='Accuracy', color='#58a6ff')
    ax.bar(x + width/2, f1s,  width, label='Macro F1',  color='#3fb950')
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.set_title('Cross-Dataset Comparison', fontsize=14, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.2, axis='y')
    plt.tight_layout()
    plt.savefig('/content/cross_dataset_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()


## How to train on a second dataset

1. Scroll back up to **Cell 2 (Configuration)**
2. Change `DATASET_NAME = 'tii-ssrc-23'` to one of:
   <!-- - `'cicids-2017'` -->
   - `'nsl-kdd'`
   - `'custom'` (then upload your CSV via the Files panel and edit `CUSTOM_CSV_PATH`)
3. Run cells 3 through 10 again. Cell 11 will then show both datasets side by side.

**Tip:** You don't need to re-run cells 1, 6, or any earlier cells — they don't depend on the dataset choice.
